In [ ]:
import io
import contextlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score, f1_score)
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score, roc_auc_score, recall_score, precision_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    BaggingClassifier
)

from sklearn.linear_model import (
    LogisticRegression,
    RidgeClassifier,
    Perceptron,
    SGDClassifier,
    PassiveAggressiveClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

In [ ]:
df = df = pd.read_csv('https://raw.githubusercontent.com/muajnstu/Large_Scale_Implementation_of_DSK_Chain/refs/heads/main/filtered%20data/Saimon%20and%20Roni/Clustered_loan_data_Processed.csv')

X = df.drop(columns=['loan_status'])
y = df['loan_status']

In [ ]:

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
print("Class distribution after ros:", pd.Series(y_resampled).value_counts())

Class distribution after ros: loan_status
10    8510
0     8510
6     8510
8     8510
7     8510
9     8510
2     8510
5     8510
4     8510
3     8510
1     8510
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=46, stratify=y_resampled
)


In [ ]:
def print_metrics(y_true, y_pred, y_prob=None):
    cm = confusion_matrix(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    num_classes = cm.shape[0]

    if num_classes == 2:
        TN, FP, FN, TP = cm.ravel()
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        gmean = np.sqrt(specificity * sensitivity)
        type1 = FP / (FP + TN) if (FP + TN) > 0 else 0
        type2 = FN / (TP + FN) if (TP + FN) > 0 else 0
        fmeasure = f1_score(y_true, y_pred, pos_label=1)
        auc = 0
        if y_prob is not None and hasattr(y_prob, "shape") and y_prob.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_prob[:, 1])
            except Exception:
                auc = 0
    else:
        TP = np.diag(cm)
        FP = np.sum(cm, axis=0) - TP
        FN = np.sum(cm, axis=1) - TP
        TN = np.sum(cm) - (FP + FN + TP)

        specificity = np.mean([
            TN[i] / (TN[i] + FP[i]) if (TN[i] + FP[i]) > 0 else 0 for i in range(num_classes)
        ])
        sensitivity = np.mean([
            TP[i] / (TP[i] + FN[i]) if (TP[i] + FN[i]) > 0 else 0 for i in range(num_classes)
        ])
        gmean = np.sqrt(specificity * sensitivity)
        type1 = np.mean([
            FP[i] / (FP[i] + TN[i]) if (FP[i] + TN[i]) > 0 else 0 for i in range(num_classes)
        ])
        type2 = np.mean([
            FN[i] / (TP[i] + FN[i]) if (TP[i] + FN[i]) > 0 else 0 for i in range(num_classes)
        ])
        fmeasure = f1_score(y_true, y_pred, average='macro')
        auc = 0
        if y_prob is not None and hasattr(y_prob, "shape") and y_prob.shape[1] > 1:
            try:
                auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
            except Exception:
                auc = 0

    print(f"Accuracy      : {accuracy:.4f}")
    print(f"Sensitivity   : {sensitivity:.4f}")
    print(f"Specificity   : {specificity:.4f}")
    print(f"G-Mean        : {gmean:.4f}")
    print(f"Type I Error  : {type1:.4f}")
    print(f"Type II Error : {type2:.4f}")
    print(f"F1 Score      : {fmeasure:.4f}")
    print(f"AUROC         : {auc:.4f}")

In [ ]:
def run_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    try:
        y_prob = model.predict_proba(X_test)
    except Exception:
        y_prob = None
    print(f"\nModel: {name}")
    print_metrics(y_test, y_pred, y_prob)

In [ ]:
ml_models = {
    # Original models
    "RandomForest": RandomForestClassifier(random_state=42),
    "ExtraTrees": ExtraTreesClassifier(random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=10000, random_state=42, solver='saga'),
    "RidgeClassifier": RidgeClassifier(random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "NaiveBayes": GaussianNB(),
    "Perceptron": Perceptron(random_state=42),
    "SGDClassifier": SGDClassifier(random_state=42),
    #"KNN": KNeighborsClassifier(n_neighbors=3),
    "PassiveAggressive": PassiveAggressiveClassifier(random_state=42),
    #"RBFSVM": SVC(kernel='rbf', probability=True, random_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "LightGBM": LGBMClassifier(verbosity=-1, random_state=42),


    "VotingSoft": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42))
        ],
        voting='soft',
        n_jobs=-1
    ),


    "VotingHard": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42))
        ],
        voting='hard',
        n_jobs=-1
    ),


    "VotingWeighted": VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        voting='soft',
        weights=[2, 3, 2, 3, 3],
        n_jobs=-1
    ),


    # ── Stacking 1: Tree ensembles → Logistic Regression meta ──────────────
    "Stacking_LR": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False))
        ],
        final_estimator=LogisticRegression(max_iter=10000, random_state=42, solver='saga'),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),


    # ── Stacking 2: Linear/probabilistic bases → LightGBM meta ─────────────
    "Stacking_LGBM": StackingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=10000, random_state=42, solver='saga')),
            ('lda', LinearDiscriminantAnalysis()),
            ('qda', QuadraticDiscriminantAnalysis()),
            ('nb', GaussianNB())
        ],
        final_estimator=LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),


    # ── Stacking 3: Weak/online learners → GradientBoosting meta ───────────
    "Stacking_GB": StackingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42)),
            ('sgd', SGDClassifier(random_state=42, loss='modified_huber')),  # modified_huber enables predict_proba
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42))
        ],
        final_estimator=GradientBoostingClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),


    # ── Stacking 4: Mixed heterogeneous bases → RandomForest meta ──────────
    "Stacking_RF": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB()),
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42))
        ],
        final_estimator=RandomForestClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),
    # ── Stacking 5: XGB-heavy diverse bases → ExtraTrees meta ─────────────
    "Stacking_ET": StackingClassifier(
        estimators=[
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('lr', LogisticRegression(max_iter=10000, random_state=42, solver='saga')),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB())
        ],
        final_estimator=ExtraTreesClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    ),
}

In [ ]:
ml_models["LDA"] = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
ml_models["QDA"] = QuadraticDiscriminantAnalysis(reg_param=0.1)

for name, model in ml_models.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: RandomForest
Accuracy      : 0.9742
Sensitivity   : 0.9742
Specificity   : 0.9974
G-Mean        : 0.9857
Type I Error  : 0.0026
Type II Error : 0.0258
F1 Score      : 0.9741
AUROC         : 0.9998

Model: ExtraTrees
Accuracy      : 0.9756
Sensitivity   : 0.9756
Specificity   : 0.9976
G-Mean        : 0.9865
Type I Error  : 0.0024
Type II Error : 0.0244
F1 Score      : 0.9756
AUROC         : 0.9999

Model: Bagging
Accuracy      : 0.9738
Sensitivity   : 0.9738
Specificity   : 0.9974
G-Mean        : 0.9855
Type I Error  : 0.0026
Type II Error : 0.0262
F1 Score      : 0.9737
AUROC         : 0.9985

Model: GradientBoosting
Accuracy      : 0.9118
Sensitivity   : 0.9118
Specificity   : 0.9912
G-Mean        : 0.9507
Type I Error  : 0.0088
Type II Error : 0.0882
F1 Score      : 0.9114
AUROC         : 0.9973

Model: LogisticRegression
Accuracy      : 0.1432
Sensitivity   : 0.1432
Specificity   : 0.9143
G-Mean        : 0.3619
Type I Error  : 0.0857
Type II Error : 0.8568
F1 Score      : 0.

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Model: Stacking_LR
Accuracy      : 0.9878
Sensitivity   : 0.9878
Specificity   : 0.9988
G-Mean        : 0.9932
Type I Error  : 0.0012
Type II Error : 0.0122
F1 Score      : 0.9877
AUROC         : 0.9999


In [ ]:
# ── Stacking 2: Linear/probabilistic bases → LightGBM meta ─────────────
stacking_2 = {"Stacking_LGBM": StackingClassifier(
    estimators=[
        ('lda', LinearDiscriminantAnalysis()),
        ('lr', LogisticRegression(max_iter=5000, random_state=42, solver='saga')),
        ('QDA', QuadraticDiscriminantAnalysis(reg_param=0.01)),
        ('nb', GaussianNB())
    ],
    final_estimator=LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42),
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1
   )
}

for name, model in stacking_2.items():
    run_model(name, model, X_train, X_test, y_train, y_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Model: Stacking_LGBM
Accuracy      : 0.9185
Sensitivity   : 0.9185
Specificity   : 0.9919
G-Mean        : 0.9545
Type I Error  : 0.0081
Type II Error : 0.0815
F1 Score      : 0.9182
AUROC         : 0.9974


In [ ]:
stacking_3 = {
    "Stacking_GB": StackingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42)),
            ('sgd', SGDClassifier(random_state=42, loss='modified_huber')),
            ('et', ExtraTreesClassifier(n_estimators=100, random_state=42))
        ],
        final_estimator=GradientBoostingClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_3.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: Stacking_GB
Accuracy      : 0.9883
Sensitivity   : 0.9883
Specificity   : 0.9988
G-Mean        : 0.9936
Type I Error  : 0.0012
Type II Error : 0.0117
F1 Score      : 0.9883
AUROC         : 0.9998


In [ ]:
stacking_4 = {
    "Stacking_RF": StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('lgbm', LGBMClassifier(verbosity=-1, n_estimators=100, random_state=42)),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB()),
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('bag', BaggingClassifier(random_state=42))
        ],
        final_estimator=RandomForestClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_4.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: Stacking_RF
Accuracy      : 0.9876
Sensitivity   : 0.9876
Specificity   : 0.9988
G-Mean        : 0.9932
Type I Error  : 0.0012
Type II Error : 0.0124
F1 Score      : 0.9876
AUROC         : 0.9997


In [ ]:
stacking_5 = {
    "Stacking_ET": StackingClassifier(
        estimators=[
            ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', use_label_encoder=False)),
            ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
            ('lr', LogisticRegression(max_iter=5000, random_state=42, solver='saga')),
            ('lda', LinearDiscriminantAnalysis()),
            ('nb', GaussianNB())
        ],
        final_estimator=ExtraTreesClassifier(n_estimators=100, random_state=42),
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )
}
for name, model in stacking_5.items():
    run_model(name, model, X_train, X_test, y_train, y_test)


Model: Stacking_ET
Accuracy      : 0.9736
Sensitivity   : 0.9736
Specificity   : 0.9974
G-Mean        : 0.9854
Type I Error  : 0.0026
Type II Error : 0.0264
F1 Score      : 0.9736
AUROC         : 0.9993


In [ ]:
'''buffer = io.StringIO()

with contextlib.redirect_stdout(buffer):
    for name, model in ml_models.items():
        run_model(name, model, X_train, X_test, y_train, y_test)

output_text = buffer.getvalue()

rows = []
current_model = None

for line in output_text.splitlines():
    line = line.strip()

    if line.startswith("Model:"):
        current_model = {"Model": line.split(":", 1)[1].strip()}
        rows.append(current_model)

    elif ":" in line and current_model is not None:
        key, value = line.split(":", 1)
        try:
            current_model[key.strip()] = float(value.strip())
        except ValueError:
            pass

df = pd.DataFrame(rows)
df.to_csv("model_outputs.csv", index=False)

print("Saved model_outputs.csv")'''

'buffer = io.StringIO()\n\nwith contextlib.redirect_stdout(buffer):\n    for name, model in ml_models.items():\n        run_model(name, model, X_train, X_test, y_train, y_test)\n\noutput_text = buffer.getvalue()\n\nrows = []\ncurrent_model = None\n\nfor line in output_text.splitlines():\n    line = line.strip()\n\n    if line.startswith("Model:"):\n        current_model = {"Model": line.split(":", 1)[1].strip()}\n        rows.append(current_model)\n\n    elif ":" in line and current_model is not None:\n        key, value = line.split(":", 1)\n        try:\n            current_model[key.strip()] = float(value.strip())\n        except ValueError:\n            pass\n\ndf = pd.DataFrame(rows)\ndf.to_csv("model_outputs.csv", index=False)\n\nprint("Saved model_outputs.csv")'

**Outputs are saved manually**